# GRPO Training on API Debug Environment

Trains Qwen 0.5B to debug malformed API requests using reward signals from the live HF Space.
Includes 6-level curriculum learning: easy -> classify -> medium -> headers -> response -> hard.

**Requirements**: Free Colab T4 GPU

In [ ]:
# Cell 1: Install dependencies
!pip install -q trl>=0.26.0 transformers torch datasets openenv-core openai

In [ ]:
# Cell 2: Clone repo
!git clone https://github.com/Avi-chauhan/api-debug-env.git
%cd api-debug-env

In [ ]:
# Cell 3: Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"BF16: {torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}")

In [ ]:
# Cell 4: Quick environment connectivity test
import asyncio
import sys
sys.path.insert(0, '.')
from client import APIDebugEnv
from models import APIDebugAction

async def test_env():
    env = APIDebugEnv(base_url='https://avichauhan-api-debug-env.hf.space')
    result = await env.reset(task='easy')
    print(f'Connected! Task: {result.observation.task}')
    print(f'API: {result.observation.api_name}')
    print(f'Error count: {result.observation.error_count}')
    result = await env.step(APIDebugAction(error_type='missing_required_field', affected_fields=['email']))
    print(f'Reward: {result.reward}, Done: {result.done}')
    await env.close()

await test_env()

In [ ]:
# Cell 5: Run GRPO training
# This will take ~30-60 minutes on a T4 GPU
!python training/train.py

In [ ]:
# Cell 6: Upload trained model to HuggingFace
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path='./outputs/api-debug-grpo',
    repo_id='avichauhan/api-debug-grpo-qwen-0.5b',
    repo_type='model',
    create_pr=False,
)
print('Model uploaded to: https://huggingface.co/avichauhan/api-debug-grpo-qwen-0.5b')